# Lab Work - 9.3

## Q.1 Hyperplanes, Margins & Support Vectors

### 01 Define a hyperplane...

A hyperplane in d-dimensional space is the set of points x satisfying w·x + b = 0, where w is the normal vector and b is the bias.

In 2D, it's a line; in 3D, a plane.

The normal vector w is perpendicular to the hyperplane and points in the direction of increasing decision value.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm

# Simple example in 2D
plt.figure(figsize=(8,6))
x = np.array([[2,3], [1,1], [3,4]])
y = np.array([1, -1, 1])

clf = svm.SVC(kernel='linear', C=100)
clf.fit(x, y)

# Plot hyperplane etc. (add more code as needed)
print('w:', clf.coef_)
print('b:', clf.intercept_)

### 02 Define distance to hyperplane...

Signed distance from point x_i to hyperplane: (w·x_i + b) / ||w||

For example with w=(1,1), b=-1, x=(2,3), y=+1:

Compute functional and geometric margin.

In [ ]:
w = np.array([1,1])
b = -1
x = np.array([2,3])
functional_margin = 1 * (np.dot(w, x) + b)
geometric_margin = functional_margin / np.linalg.norm(w)
print('Functional margin:', functional_margin)
print('Geometric margin:', geometric_margin)

Continue with other questions similarly...

## Q.2 Hard Margin vs Soft Margin

Explanation of hard-margin failure, soft-margin with slack variables, etc.

In [ ]:
# Example with soft margin
from sklearn.svm import SVC
# Dataset with outlier
# Code to demonstrate

## Q.3 The Kernel Trick

### Understanding the Kernel Trick

The kernel trick is a powerful technique that allows SVMs to find non-linear decision boundaries without explicitly computing high-dimensional feature transformations.

**Key Idea:** Instead of computing φ(x) explicitly, we compute K(x_i, x_j) = φ(x_i)·φ(x_j) directly, avoiding the "curse of dimensionality."

**Why it matters:**
- Computing φ(x) can be prohibitively expensive for high dimensions
- The dot product K(x_i, x_j) is much cheaper to compute
- We never need to know φ(x) explicitly — only the kernel function K

**Common Kernels:**
1. **Linear:** K(x_i, x_j) = x_i·x_j
2. **Polynomial:** K(x_i, x_j) = (x_i·x_j + c)^d
3. **RBF (Radial Basis Function):** K(x_i, x_j) = exp(-γ||x_i - x_j||²)
4. **Sigmoid:** K(x_i, x_j) = tanh(κ(x_i·x_j) + θ)

**How it works:** When we use a kernel K, the SVM decision function becomes:
f(x) = sign(Σ α_i y_i K(x_i, x) + b)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_moons, make_circles
from mpl_toolkits.mplot3d import Axes3D

# Generate non-linearly separable data
X_moons, y_moons = make_moons(n_samples=100, noise=0.1, random_state=42)
X_circles, y_circles = make_circles(n_samples=100, noise=0.05, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Test different kernels on moons dataset
kernels = ['linear', 'poly', 'rbf']
datasets = [('Moons', X_moons, y_moons), ('Circles', X_circles, y_circles)]

for row, (name, X, y) in enumerate(datasets):
    for col, kernel in enumerate(kernels):
        clf = SVC(kernel=kernel, C=1.0, gamma='scale')
        if kernel == 'poly':
            clf = SVC(kernel=kernel, degree=3, C=1.0, gamma='scale')
        
        clf.fit(X, y)
        
        # Create mesh
        h = 0.02
        x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
        y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                             np.arange(y_min, y_max, h))
        
        Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        
        # Plot
        ax = axes[row, col]
        ax.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
        ax.scatter(X[y==1, 0], X[y==1, 1], c='red', edgecolors='k', label='Class 1', s=50)
        ax.scatter(X[y==0, 0], X[y==0, 1], c='blue', edgecolors='k', label='Class 0', s=50)
        
        # Mark support vectors
        if hasattr(clf, 'support_vectors_'):
            ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], 
                      s=200, linewidth=1.5, facecolors='none', edgecolors='green', label='Support Vectors')
        
        ax.set_title(f'{name} - {kernel.upper()} Kernel')
        ax.set_xlim(xx.min(), xx.max())
        ax.set_ylim(yy.min(), yy.max())
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Example: Demonstrating kernel computation
print("\n--- Kernel Computation Examples ---")
x1 = np.array([1, 2])
x2 = np.array([3, 4])

print(f"x1 = {x1}, x2 = {x2}")
print(f"Linear kernel:    K(x1,x2) = {np.dot(x1, x2)}")
print(f"Polynomial kernel (d=2): K(x1,x2) = {(np.dot(x1, x2) + 1)**2}")
print(f"RBF kernel (γ=0.5):      K(x1,x2) = {np.exp(-0.5 * np.sum((x1-x2)**2)):.4f}")

## Q.4 Deep Geometric Intuition

### Geometric Intuition Behind SVMs

**1. The Separation Principle**
- SVMs find the hyperplane that maximizes the minimum distance (margin) to the nearest points
- This is equivalent to finding the widest "street" that separates the classes
- The points on the boundary of this margin are the **support vectors**

**2. Maximum Margin Concept**
- Hard-margin SVM: perfectly separates classes with maximum margin
- Soft-margin SVM: allows some misclassification to achieve better generalization (margin vs. regularization trade-off)
- The margin is: γ = 1/||w|| (proportional to the geometric distance between classes)

**3. Non-linear Separation via Kernel Trick**
- Linear kernels: find linear decision boundaries
- Non-linear kernels (RBF, Polynomial): implicitly map data to higher dimensions where linear separation is possible
- RBF kernel creates "radial bumps" around each support vector, allowing complex curved boundaries
- The kernel trick avoids explicit high-dimensional computation

**4. Support Vectors as Decision Makers**
- Only support vectors matter for the final decision: f(x) = sign(Σ α_i y_i K(x_i, x) + b)
- Non-support vectors have α_i = 0 and don't contribute
- This sparsity makes SVMs efficient and helps with generalization (few training points needed to define the boundary)

**5. Decision Boundary Properties**
- The decision boundary is determined by the support vectors alone
- Moving non-support vectors (outside the margin) doesn't change the model
- The model is robust to outliers: only support vectors affect it
- α_i values reflect the "importance" of each support vector

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_blobs

# Create synthetic data
X, y = make_blobs(n_samples=50, n_features=2, centers=2, random_state=42, cluster_std=1.5)

# Train SVM with different C values
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
C_values = [0.1, 1, 10]

for idx, C in enumerate(C_values):
    clf = SVC(kernel='linear', C=C, random_state=42)
    clf.fit(X, y)
    
    # Create mesh for decision boundary
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax = axes[idx]
    
    # Plot decision boundary
    ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
    # Plot margins
    ax.contour(xx, yy, Z, levels=[1, -1], linewidths=1, colors='gray', linestyles='--')
    
    # Shade regions
    ax.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.3)
    
    # Plot data points
    ax.scatter(X[y==0, 0], X[y==0, 1], c='red', edgecolors='k', s=100, label='Class 0')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', edgecolors='k', s=100, label='Class 1')
    
    # Highlight support vectors
    sv_indices = clf.support_
    ax.scatter(X[sv_indices, 0], X[sv_indices, 1], s=400, linewidth=2, 
              facecolors='none', edgecolors='green', label='Support Vectors')
    
    n_support_vectors = len(clf.support_)
    margin = 1 / np.linalg.norm(clf.coef_)
    
    ax.set_title(f'C={C}\nSupport Vectors: {n_support_vectors}, Margin: {margin:.3f}')
    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of Regularization Parameter C on Margin and Support Vectors', fontsize=14)
plt.tight_layout()
plt.show()

# Demonstrate sparsity and α_i values
print("\n--- Support Vectors and Their Importance ---")
print(f"Total training samples: {len(X)}")
print(f"Number of support vectors: {len(clf.support_)}")
print(f"Percentage: {len(clf.support_)/len(X)*100:.1f}%")
print(f"\nSupport vector indices: {clf.support_}")
print(f"\nDual coefficients (α_i * y_i):")
for i, coef in enumerate(clf.dual_coef_[0]):
    print(f"  α_{clf.support_[i]} * y = {coef:.4f}")
print(f"\nBias (b): {clf.intercept_[0]:.4f}")
print(f"Weight vector (w): {clf.coef_[0]}")
print(f"Margin = 1/||w|| = {1/np.linalg.norm(clf.coef_[0]):.4f}")